In [1]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install 'litgpt[extra]'

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [4]:
import os
import json
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

def chunk_text(text, chunk_size=400, overlap=50):
    """
    Splits text into overlapping chunks for better finetuning.
    - chunk_size: number of words per chunk
    - overlap: number of words to overlap between chunks (gives context continuity)
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # slide forward with overlap

        # Stop if remaining words are too few to be useful
        if len(words) - start < 50:
            break

    return chunks

def generate_basic_summary(text, num_sentences=5):
    """
    Generates a basic extractive summary by picking the most
    'important' sentences using simple word frequency scoring.
    No external libraries needed — pure Python.
    """
    # Split into sentences
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if len(s.strip()) > 30]

    if not sentences:
        return text[:300]  # fallback: just return first 300 chars

    # Count word frequencies (ignore short/common words)
    stopwords = {"the", "a", "an", "is", "in", "it", "of", "and", "to", "was",
                 "for", "on", "are", "as", "at", "be", "by", "this", "that",
                 "with", "from", "or", "but", "not", "have", "had", "has"}

    word_freq = {}
    for sentence in sentences:
        for word in sentence.lower().split():
            word = word.strip(".,!?;:()")
            if word not in stopwords and len(word) > 3:
                word_freq[word] = word_freq.get(word, 0) + 1

    # Score each sentence by the sum of its word frequencies
    sentence_scores = {}
    for sentence in sentences:
        score = 0
        for word in sentence.lower().split():
            word = word.strip(".,!?;:()")
            score += word_freq.get(word, 0)
        sentence_scores[sentence] = score

    # Pick top N sentences and return them in original order
    top_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:num_sentences]
    summary = ". ".join([s for s in sentences if s in top_sentences])
    return summary + "."

def process_documents(input_dir="./documents", output_file="training_data.json"):
    """
    Processes all PDF files in a directory, extracts text, chunks it,
    generates extractive summaries as outputs, and saves to JSON.
    """
    training_data = []

    if not os.path.exists(input_dir):
        print(f"Error: Input directory '{input_dir}' not found.")
        return

    print(f"Processing PDFs from '{input_dir}'...")

    for filename in os.listdir(input_dir):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(input_dir, filename)
            print(f"\n  Extracting text from: {filename}")

            try:
                full_text = extract_text_from_pdf(pdf_path)
                print(f"    → Extracted {len(full_text.split())} words")

                # Split into chunks so each chunk becomes one training sample
                chunks = chunk_text(full_text, chunk_size=400, overlap=50)
                print(f"    → Created {len(chunks)} chunks")

                for idx, chunk in enumerate(chunks):
                    instruction = (
                        f"Summarize the key information from section {idx+1} "
                        f"of the document titled '{filename}'."
                    )

                    # Generate an extractive summary of this chunk as the output
                    output = generate_basic_summary(chunk, num_sentences=3)

                    training_data.append({
                        "instruction": instruction,
                        "input": chunk,
                        "output": output  # ← now populated with real content!
                    })

            except Exception as e:
                print(f"    Error processing {filename}: {e}")

    if training_data:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(training_data, f, indent=4, ensure_ascii=False)

        print(f"\n✅ Done! Created {len(training_data)} training samples → '{output_file}'")
        print("\nPreview of first entry:")
        print(f"  instruction : {training_data[0]['instruction']}")
        print(f"  input       : {training_data[0]['input'][:100]}...")
        print(f"  output      : {training_data[0]['output'][:150]}...")
    else:
        print("No PDF files found or processed.")

if __name__ == "__main__":
    process_documents()

Processing PDFs from './documents'...

  Extracting text from: 3545008.3545087.pdf
    → Extracted 8450 words
    → Created 25 chunks

  Extracting text from: applsci-12-02160-v2.pdf
    → Extracted 6609 words
    → Created 19 chunks

  Extracting text from: 3458817.3476223.pdf
    → Extracted 14437 words
    → Created 42 chunks

  Extracting text from: Paper1.pdf
    → Extracted 4012 words
    → Created 12 chunks

  Extracting text from: DBA_Residency_hsampatirao.pdf
    → Extracted 575 words
    → Created 2 chunks

  Extracting text from: Paper2.pdf
    → Extracted 3617 words
    → Created 11 chunks

  Extracting text from: Hariprasad_Sampatirao_Draft_Chapter1.pdf
    → Extracted 8649 words
    → Created 25 chunks

  Extracting text from: atc23-weng.pdf
    → Extracted 11492 words
    → Created 33 chunks

  Extracting text from: 3638757.pdf
    → Extracted 18766 words
    → Created 54 chunks

  Extracting text from: Walsh_Dissertation_Data Readiness.pdf
    → Extracted 5171 words
   

In [5]:
import subprocess
import os

# --- Configuration ---
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 1.1B params, fits T4 without quantization

# Path to your processed JSON data file
TRAINING_DATA_PATH = "training_data.json"

# Directory where the finetuned model checkpoints will be saved
OUTPUT_DIR = "out/tinyllama-finetuned"

FINETUNE_COMMAND = [
    "litgpt",
    "finetune",
    BASE_MODEL,
    "--data", "JSON",
    f"--data.json_path={TRAINING_DATA_PATH}",
    "--data.val_split_fraction=0.1",
    f"--out_dir={OUTPUT_DIR}",
    "--train.micro_batch_size=2",    # can afford slightly larger batch with 1.1B model
    "--train.global_batch_size=8",
    "--train.epochs=3",
    "--device=auto",
    "--train.max_seq_length=512",
    "--precision=16-mixed",          # float16 with loss scaling, no quantization needed
    "--lora_r=8",
    "--lora_alpha=16",
    "--lora_query=true",
    "--lora_value=true",
    "--lora_key=true",
]

def run_finetuning():
    print("--- Starting LLM Finetuning ---")
    print(f"Base model: {BASE_MODEL}")
    print(f"Training data: {TRAINING_DATA_PATH}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Finetuning command: {' '.join(FINETUNE_COMMAND)}\n")

    if not os.path.exists(TRAINING_DATA_PATH):
        print(f"Error: Training data file '{TRAINING_DATA_PATH}' not found.")
        print("Please ensure you have processed your PDFs and created this file.")
        return

    try:
        process = subprocess.Popen(
            FINETUNE_COMMAND,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

        for line in process.stdout:
            print(line, end="")

        process.wait()

        if process.returncode == 0:
            print("\n--- Finetuning completed successfully! ---")
            print(f"Model checkpoints saved to: {OUTPUT_DIR}")
            print("You can now test or deploy your finetuned model.")
        else:
            print(f"\n--- Finetuning failed with return code {process.returncode} ---")

    except FileNotFoundError:
        print("Error: 'litgpt' command not found.")
        print("Install via: pip install 'litgpt[extra]'")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    run_finetuning()

--- Starting LLM Finetuning ---
Base model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Training data: training_data.json
Output directory: out/tinyllama-finetuned
Finetuning command: litgpt finetune TinyLlama/TinyLlama-1.1B-Chat-v1.0 --data JSON --data.json_path=training_data.json --data.val_split_fraction=0.1 --out_dir=out/tinyllama-finetuned --train.micro_batch_size=2 --train.global_batch_size=8 --train.epochs=3 --device=auto --train.max_seq_length=512 --precision=16-mixed --lora_r=8 --lora_alpha=16 --lora_query=true --lora_value=true --lora_key=true

Using 16-bit Automatic Mixed Precision (AMP)
Seed set to 1337
{'access_token': None,
 'checkpoint_dir': PosixPath('checkpoints/TinyLlama/TinyLlama-1.1B-Chat-v1.0'),
 'data': JSON(json_path=PosixPath('training_data.json'),
              mask_prompt=False,
              val_split_fraction=0.1,
              prompt_style=<litgpt.prompts.Alpaca object at 0x720c11510e30>,
              ignore_index=-100,
              seed=42,
              num_work

In [6]:
!pip install "nbconvert[webpdf]"
!playwright install chromium

In [7]:
!jupyter nbconvert --to webpdf llm_finetuning.ipynb --allow-chromium-download

[NbConvertApp] Converting notebook llm_finetuning.ipynb to webpdf
[NbConvertApp] Building PDF
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nbconvert/exporters/webpdf.py", line 110, in main
    browser = await chromium.launch(
              ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/playwright/async_api/_generated.py", line 14547, in launch
    await self._impl_obj.launch(
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/playwright/_impl/_browser_type.py", line 97, in launch
    await self._channel.send(
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/playwright/_impl/_connection.py", line 69, in send
    return await self._connection.wrap_api_call(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/playwright/_impl/_connection.py", line 559, in

In [8]:
!jupyter nbconvert --to html llm_finetuning.ipynb

[NbConvertApp] Converting notebook llm_finetuning.ipynb to html
[NbConvertApp] Writing 363014 bytes to llm_finetuning.html
